# Phase 4a: Test Before Deploying
Run the same detection logic `app.py` uses, on a real VisDrone image, right here in the notebook.

**Why this comes first:** if something's wrong (boxes shifted, nothing detected, a crash), it's much faster to find and fix here than after uploading to Hugging Face and waiting for a Space to build.

**Input needed:** none new. This reuses Phase 3's exported `.onnx` file plus a sample image from the VisDrone dataset, both should already be reachable from `/kaggle/input` or `/kaggle/working` depending on your session setup.

In [1]:
!pip install -q onnxruntime pillow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 42.8 MB/s eta 0:00:00


In [2]:
"""Find a VisDrone validation image to test on."""
from pathlib import Path


def find_test_image(search_roots: tuple[str, ...] = (".", "/kaggle/input", "/kaggle/working")) -> Path:
    for root in search_roots:
        if Path(root).exists():
            matches = list(Path(root).rglob("images/val/*.jpg"))
            if matches:
                return matches[0]
    raise FileNotFoundError(
        "No VisDrone val image found under " + str(search_roots) + ". "
        "Set test_image_path manually to any .jpg on disk instead."
    )


test_image_path = find_test_image()
print(f"Testing on: {test_image_path}")

FileNotFoundError: No VisDrone val image found under ('.', '/kaggle/input', '/kaggle/working'). Set test_image_path manually to any .jpg on disk instead.

In [ ]:
"""
Load the exported ONNX model and run the exact same detection logic app.py
uses. If this looks right, app.py will look right too, since it's the same code.
"""
import time
import numpy as np
import onnxruntime as ort
from PIL import Image, ImageDraw
from IPython.display import display


def find_onnx_model(model_name: str, search_roots: tuple[str, ...] = (".", "/kaggle/input", "/kaggle/working")) -> Path:
    matches = []
    for root in search_roots:
        if Path(root).exists():
            matches.extend(Path(root).rglob(model_name))
    matches = sorted(matches, key=lambda p: p.stat().st_mtime, reverse=True)
    if not matches:
        raise FileNotFoundError(f"No '{model_name}' found under {search_roots}, run Phase 3 first.")
    return matches[0]


MODEL_PATH = find_onnx_model("yolo26n_best.onnx")
print(f"Using model: {MODEL_PATH}")

IMG_SIZE = 640
CONF_THRESHOLD = 0.25
CLASS_NAMES = [
    "pedestrian", "people", "bicycle", "car", "van",
    "truck", "tricycle", "awning-tricycle", "bus", "motor",
]

session = ort.InferenceSession(str(MODEL_PATH), providers=["CPUExecutionProvider"])
INPUT_NAME = session.get_inputs()[0].name


def preprocess(image: Image.Image) -> tuple[np.ndarray, float, float]:
    orig_w, orig_h = image.size
    resized = image.convert("RGB").resize((IMG_SIZE, IMG_SIZE))
    arr = np.asarray(resized, dtype=np.float32) / 255.0
    arr = arr.transpose(2, 0, 1)[None, ...]
    return arr, orig_w / IMG_SIZE, orig_h / IMG_SIZE


def detect(image: Image.Image) -> tuple[Image.Image, str]:
    tensor, scale_x, scale_y = preprocess(image)
    start = time.perf_counter()
    outputs = session.run(None, {INPUT_NAME: tensor})
    latency_ms = (time.perf_counter() - start) * 1000

    detections = outputs[0][0]
    detections = detections[detections[:, 4] >= CONF_THRESHOLD]

    annotated = image.convert("RGB").copy()
    draw = ImageDraw.Draw(annotated)
    for x1, y1, x2, y2, score, cls_id in detections:
        box = [x1 * scale_x, y1 * scale_y, x2 * scale_x, y2 * scale_y]
        label = CLASS_NAMES[int(cls_id)] if int(cls_id) < len(CLASS_NAMES) else str(int(cls_id))
        draw.rectangle(box, outline="lime", width=2)
        draw.text((box[0], max(box[1] - 12, 0)), f"{label} {score:.2f}", fill="lime")

    return annotated, f"{len(detections)} objects, {latency_ms:.1f} ms"


test_image = Image.open(test_image_path)
annotated, stats = detect(test_image)
print(stats)
display(annotated)

## Check the image above

**Looks right:** green boxes sitting on actual vehicles, pedestrians, etc., roughly where you'd expect. You're ready for Step 2, deploy to HF Spaces.

**Looks wrong:** boxes shifted off the objects, or none appear at all, that's the letterbox/preprocessing mismatch flagged in the code comments. Paste this output back to me before deploying, it's a five-minute fix here versus a confusing debug session on a live Space.

## Step 2: Deploy

1. Download the `.onnx` file this notebook used (path printed above) from Kaggle's Output tab.
2. Create a Space at huggingface.co/new-space, SDK: Gradio, hardware: CPU basic.
3. Upload `app.py`, `requirements.txt`, and the `.onnx` file renamed to `model.onnx`.
4. Space builds automatically, you get a public URL.